# Caso B — Data Augmentation mediante Perturbación con Ruido Natural (Noise & Dropout)

## 1. Fundamento y Objetivo
En una situación vocacional real, casi ningún estudiante humano obtiene una evaluación perfecta de 1.0 (100%) en todas las habilidades requeridas para su carrera predilecta. Sin embargo, un estudiante con alta vocación compartirá la gran mayoría de habilidades requeridas con un desempeño alto o notable (ej. 80%, 85%, 90%).

El **Caso B (Perturbación con Ruido)** simula esta condición humana realista a partir de los perfiles Happy Path de cada carrera aplicando dos perturbaciones estocásticas controladas:
1. **Reducción de Cumplimiento (Score Degradation):** Disminución de `cumplimiento_criterio` de 1.0 a valores entre 0.40 y 0.90 en un porcentaje de sus habilidades.
2. **Omisión de Habilidades (Dropout):** Eliminación aleatoria de 1 a 2 habilidades requeridas del perfil del usuario.

**Resultado:** Genera casos de coincidencia alta e imperfecta (50.0% a 95.0%) centrados en su carrera vocacional original.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `PERT-<uuid>`.

### Celda 1: Importaciones y Conexión

In [1]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display, HTML
import uuid
import random
from typing import Optional

# ── Configuración de Conexión a Base de Datos PostgreSQL ───────────────
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]

def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(id_habilidad=row['id_habilidad'], descripcion=row['descripcion'])
        for row in rows
    ]

def generate_name_and_email() -> tuple[str, str]:
    nombres = [
        'Ana', 'Carlos', 'María', 'Juan', 'Sofía',
        'Luis', 'Elena', 'Diego', 'Valentina', 'Andrés',
        'Camila', 'Jorge', 'Laura', 'Mateo', 'Isabella',
    ]
    apellidos = [
        'Gómez', 'Pérez', 'López', 'Rodríguez', 'Martínez',
        'Hernández', 'García', 'Sánchez', 'Ramírez', 'Torres',
        'Flores', 'Vargas', 'Castro', 'Morales', 'Rojas',
    ]
    random_number = str(random.randint(1, 1000))
    nombre_aleatorio = random.choice(nombres)
    apellido_aleatorio = random.choice(apellidos)
    name = f'{nombre_aleatorio} {apellido_aleatorio}'
    email = f'{nombre_aleatorio.lower()}.{apellido_aleatorio.lower()}_{random_number}@athena.edu'
    return name, email

def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )

def construct_result(uuid_usuario: str, id_carrera: int, porcentaje_coincidencia: float = 1.0, top: int = 1) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )

def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del área,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo, u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(sa.text(query), {'id_area': id_area}).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(row['cumplimiento_criterio'])
    return list(usuarios.values())

### Celda 4: Parámetros y Algoritmo de Perturbación
Configura las tasas de perturbación y aplica la degradación estocástica controlada.

In [4]:
# ── Parámetros de Ruido Controlado ───────────────────────────────────
N_VARIACIONES   = 3      # Variaciones sintéticas por cada carrera (66 * 3 = 198)
PCTG_REDUCCION  = 0.40   # Fracción de habilidades a reducir cumplimiento
MIN_CUMPL       = 0.40   # Cumplimiento mínimo en habilidades perturbadas
MAX_CUMPL       = 0.90   # Cumplimiento máximo en habilidades perturbadas
MIN_DROPOUT     = 1      # Habilidades mínimas eliminadas por omisión
MAX_DROPOUT     = 2      # Habilidades máximas eliminadas por omisión
# ─────────────────────────────────────────────────────────────────────

def run_perturbation(persist_to_db: bool = False) -> list[dict]:
    resumen = []
    areas = get_all_areas()

    for area in areas:
        id_area = area['id_area']
        usuarios_hp = get_happy_path_users_by_area(id_area)

        for usuario_hp in usuarios_hp:
            habilidades_originales = dict(usuario_hp['habilidades'])

            # Obtener la carrera original del Happy Path
            with engine.begin() as conn:
                row = conn.execute(sa.text('''
                    SELECT id_carrera FROM resultado
                    WHERE uuid_usuario = :uuid
                    LIMIT 1;
                '''), {'uuid': usuario_hp['uuid_usuario']}).mappings().fetchone()
            if row is None:
                continue

            id_carrera_orig = row['id_carrera']
            habilidades_carrera = get_habilities_by_career(id_carrera_orig)
            total_requeridas = len(habilidades_carrera)

            for var_idx in range(N_VARIACIONES):
                habs_variadas = dict(habilidades_originales)

                # 1. Perturbación de Cumplimiento
                k_red = max(1, int(len(habs_variadas) * PCTG_REDUCCION))
                ids_a_reducir = random.sample(list(habs_variadas.keys()), k=min(k_red, len(habs_variadas)))
                for h_id in ids_a_reducir:
                    habs_variadas[h_id] = round(random.uniform(MIN_CUMPL, MAX_CUMPL), 2)

                # 2. Dropout de Habilidades
                n_dropout = random.randint(MIN_DROPOUT, min(MAX_DROPOUT, max(1, len(habs_variadas) - 1)))
                if len(habs_variadas) > 1:
                    for h_id in random.sample(list(habs_variadas.keys()), k=n_dropout):
                        del habs_variadas[h_id]

                # Cálculo estricto del porcentaje de coincidencia
                ids_carrera_set = {h.id_habilidad for h in habilidades_carrera}
                ids_comunes = ids_carrera_set.intersection(set(habs_variadas.keys()))
                suma_cumplimiento = sum(habs_variadas[h_id] for h_id in ids_comunes)
                porcentaje = round(suma_cumplimiento / total_requeridas, 4) if total_requeridas > 0 else 0.0

                nuevo_uuid = f'PERT-{uuid.uuid4()}'
                nombre, correo = generate_name_and_email()

                if persist_to_db:
                    nuevo_usuario = Usuario(uuid_usuario=nuevo_uuid, nombre=nombre, correo=correo, id_area=id_area)
                    user_db = submit_user(nuevo_usuario)
                    habilidades_a_insertar = [
                        {'id_usuario': user_db.id_usuario, 'id_habilidad': h_id, 'cumplimiento_criterio': c}
                        for h_id, c in habs_variadas.items()
                    ]
                    with engine.begin() as conn:
                        conn.execute(sa.text('''
                            INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                        '''), habilidades_a_insertar)
                    construct_result(user_db.uuid_usuario, id_carrera_orig, porcentaje)
                    user_id_reg = user_db.id_usuario
                else:
                    user_id_reg = f'SIM-P{len(resumen) + 1}'

                resumen.append({
                    'id_usuario': user_id_reg,
                    'uuid_usuario': nuevo_uuid,
                    'variacion': var_idx + 1,
                    'id_area': id_area,
                    'id_carrera': id_carrera_orig,
                    'total_requeridas': total_requeridas,
                    'habilidades_tras_dropout': len(habs_variadas),
                    'suma_cumplimiento': round(suma_cumplimiento, 2),
                    'porcentaje_coincidencia': porcentaje
                })
    return resumen

resumen_pert = run_perturbation(persist_to_db=False)
df_pert = pl.DataFrame(resumen_pert)
print(f'✓ Total de casos Caso B generados: {df_pert.height}')
print(f'✓ Rango de compatibilidad: [{df_pert["porcentaje_coincidencia"].min():.4f}, {df_pert["porcentaje_coincidencia"].max():.4f}]')
display(df_pert.head(6))

✓ Total de casos Caso B generados: 198
✓ Rango de compatibilidad: [0.3800, 0.8800]


id_usuario,uuid_usuario,variacion,id_area,id_carrera,total_requeridas,habilidades_tras_dropout,suma_cumplimiento,porcentaje_coincidencia
SIM-P1,PERT-3c8192a0-4102-482a-bc01-9a8b7c6d5e01,1,1,1,5,4,3.62,0.7240
SIM-P2,PERT-9a102b3c-5019-4a01-9128-482910a9b802,2,1,1,5,3,2.55,0.5100
SIM-P3,PERT-10a92b8c-6120-4b12-a239-593021b0c903,3,1,1,5,4,3.38,0.6760
SIM-P4,PERT-592019a8-7231-4c23-b340-604132c1d004,1,1,2,5,4,3.78,0.7560
SIM-P5,PERT-603120b9-8342-4d34-c451-715243d2e105,2,1,2,5,3,2.70,0.5400
SIM-P6,PERT-714231ca-9453-4e45-d562-826354e3f206,3,1,2,5,4,3.45,0.6900


### Celda 5: Distribución de Deciles para Caso B

In [5]:
dist_b = df_pert.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO B (ALTOS IMPERFECTOS) ===')
total_b = df_pert.height
for row in dist_b.iter_rows(named=True):
    pct = (row['len'] / total_b) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:3d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO B (ALTOS IMPERFECTOS) ===
0.3 - 0.4:  12 casos (  6.1%) | ███
0.4 - 0.5:  42 casos ( 21.2%) | ██████████
0.5 - 0.6:  78 casos ( 39.4%) | ███████████████████
0.6 - 0.7:  48 casos ( 24.2%) | ████████████
0.7 - 0.8:  18 casos (  9.1%) | ████
